# EDA - Bitext Customer Support Dataset

Task brief: `data/DATA_TASK.md`. Target row format: `data/schema.md`.

Goal: decide whether we can build our "blunt draft -> warm reply" training data
from this dataset. Write your findings in the markdown cell under each step.

Setup: `pip install datasets pandas matplotlib` and put `HF_TOKEN=...` in a
`.env` file in the repo root (a Read token from huggingface.co).

In [ ]:
import os, re, json, textwrap, random
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

# load HF_TOKEN from .env if present (optional for this public dataset)
env = os.path.join('..', '.env')
if os.path.exists(env):
    for line in open(env):
        if '=' in line and not line.strip().startswith('#'):
            k, v = line.strip().split('=', 1)
            os.environ.setdefault(k, v)

ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
df = ds.to_pandas()
print(df.shape)
df.head()

## 1. Load & eyeball

Look at 20 random rows. What does a typical customer message look like? A typical reply?

In [ ]:
print('columns:', list(df.columns))
print('nulls:\n', df.isna().sum(), '\n')

for _, r in df.sample(20, random_state=0).iterrows():
    print('CATEGORY :', r['category'], '| INTENT:', r['intent'], '| FLAGS:', r['flags'])
    print('CUSTOMER :', textwrap.shorten(r['instruction'], 300))
    print('RESPONSE :', textwrap.shorten(r['response'], 400))
    print('-' * 100)

**Findings (1):**

- 

## 2. Category balance

How many rows per `category`? Which are big, which are tiny or missing?

In [ ]:
counts = df['category'].value_counts()
print(counts)
counts.sort_values().plot.barh(figsize=(7, 4), title='rows per category')
plt.tight_layout(); plt.show()

print('\nintents per category:')
print(df.groupby('category')['intent'].nunique())

**Findings (2):**

- 

## 3. Lengths

Word-count histograms for the customer message and the reply. Anything empty or huge?

In [ ]:
df['msg_words'] = df['instruction'].str.split().str.len()
df['reply_words'] = df['response'].str.split().str.len()
print(df[['msg_words', 'reply_words']].describe())

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
df['msg_words'].plot.hist(bins=40, ax=ax[0], title='customer message words')
df['reply_words'].plot.hist(bins=40, ax=ax[1], title='reply words')
plt.tight_layout(); plt.show()

print('empty messages:', (df['msg_words'] == 0).sum())
print('empty replies :', (df['reply_words'] == 0).sum())

**Findings (3):**

- 

## 4. Reply quality - THE KEY QUESTION

Read ~30 replies. For each, decide: **warm & human (good)** or **stiff / templated / robotic (bad)**?
Roughly what fraction are good? Paste 3 good + 3 bad below.

The keyword check is only a rough hint - your read is what matters.

In [ ]:
warm_cues = r"\b(sorry|apolog|understand|happy to|glad to|thanks|thank you|no worries|absolutely|of course|rest assured)\b"
placeholder = r"\{\{.*?\}\}|\{%.*?%\}|\[.*?\]"

df['has_warm_cue'] = df['response'].str.contains(warm_cues, case=False, regex=True)
df['has_placeholder'] = df['response'].str.contains(placeholder, regex=True)
print('replies with a warm cue   :', df['has_warm_cue'].mean().round(3))
print('replies with a placeholder:', df['has_placeholder'].mean().round(3))

print('\n--- read these and judge for yourself ---')
for _, r in df.sample(30, random_state=1).iterrows():
    print('[', r['category'], ']')
    print(textwrap.fill(r['response'], 100))
    print('-' * 100)

**Findings (4):**

- Roughly __% of replies read as warm/human.
- Good examples:
  1. 
- Bad examples:
  1. 

## 5. Junk check

Duplicates? Leftover personal info? Template placeholders?

In [ ]:
print('exact duplicate (message, reply) pairs:', df.duplicated(['instruction', 'response']).sum())
print('duplicate customer messages          :', df['instruction'].duplicated().sum())

email = df['response'].str.contains(r"[\w.+-]+@[\w-]+\.[\w.-]+", regex=True)
digits = df['response'].str.contains(r"\d{5,}", regex=True)
print('\nreplies containing an email  :', email.sum())
print('replies containing a long number:', digits.sum())
print('replies with {{placeholders}}   :', df['has_placeholder'].sum())

df.loc[df['has_placeholder'], 'response'].head(5).tolist()

**Findings (5):**

- 

## 6. Map Bitext categories -> our 6 buckets

We need: `billing`, `technical`, `angry_complaint`, `cancellation`, `praise`, `edge_case`.
Fill the mapping, then see how many rows each bucket would get. Flag the thin ones.

In [ ]:
# EDIT THIS after you have seen the real category names in step 2.
# key = our bucket, value = list of Bitext category names that feed it.
bucket_map = {
    'billing':        ['INVOICE', 'PAYMENT', 'REFUND'],
    'technical':      ['ACCOUNT', 'DELIVERY', 'SHIPPING'],
    'angry_complaint': [],   # probably not a category here - note it
    'cancellation':   ['CANCELLATION'],
    'praise':         ['FEEDBACK'],   # check: is FEEDBACK positive, negative, or mixed?
    'edge_case':      ['CONTACT', 'NEWSLETTER'],
}

cat_to_bucket = {c: b for b, cs in bucket_map.items() for c in cs}
df['bucket'] = df['category'].map(cat_to_bucket)
print(df['bucket'].value_counts(dropna=False))
print('\nunmapped categories:', sorted(set(df.loc[df['bucket'].isna(), 'category'])))

**Findings (6):**

| our bucket | Bitext categories used | rows available | enough? |
|---|---|---|---|
| billing | | | |
| technical | | | |
| angry_complaint | | | |
| cancellation | | | |
| praise | | | |
| edge_case | | | |

## Report to send to <owner>

Fill this in and paste into the group chat / a `.md` file:

1. Total rows: __ · estimated usable: __
2. Rows available per our 6 buckets: (table from step 6)
3. Avg customer-message length: __ words · avg reply length: __ words
4. **Are the existing replies good enough to use as targets?** yes / mostly / no - and why
5. Problems: missing buckets / PII / duplicates / placeholders
6. 6 example rows pasted (3 good replies, 3 bad)
7. Dataset license + required attribution text

Then stop - we pick the Part 2 plan together based on this.